# Train Mnemo on your own knowledge (free Google Colab)

This notebook teaches **Qwen 2.5 7B** the answers Mnemo has learned, and gives you **`mnemo-7b-q4_k_m.gguf`**: your own trained model.

**Before you start**
1. In ATLAS: **Quality → Training data** to download `mnemo-training-YYYYMMDD.jsonl`.
   More data is better: approve good answers (👍), add benchmark questions, review flashcards. Aim for **100+ lines**.
2. Here in Colab: **Runtime → Change runtime type → T4 GPU → Save**.
3. Run the cells **one by one** from top to bottom (▶ button). It takes about 20–40 minutes.

**After it finishes:** put the downloaded `.gguf` in your **Downloads** folder and run
`C:\xampp\htdocs\atlas\ai\Install trained Mnemo.bat`.

_Unsloth changes often. If a cell fails, compare it with Unsloth's official "Qwen 2.5 (7B) conversational" notebook._

## 1 · Install the training tools

In [ ]:
%%capture
!pip install unsloth

## 2 · Upload your training file from ATLAS

In [ ]:
from google.colab import files
up = files.upload()            # choose mnemo-training-YYYYMMDD.jsonl
DATA = list(up.keys())[0]
n = sum(1 for _ in open(DATA, encoding='utf-8'))
print(f'{DATA}: {n} examples')
if n < 30: print('Warning: very few examples. Mnemo will learn little. Teach ATLAS more first.')

## 3 · Load Qwen 2.5 7B (4-bit, fits a free T4)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length = 2048,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth', random_state = 3407,
)

## 4 · Prepare the data (Qwen chat format)

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files = DATA, split = 'train')
ds = ds.map(lambda ex: {'text': tokenizer.apply_chat_template(ex['messages'], tokenize = False)})
print(ds[0]['text'][:600])

## 5 · Train

In [ ]:
from trl import SFTTrainer, SFTConfig
steps = max(30, min(400, len(ds) * 3 // 8))   # about 3 passes over your data
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = 'text', per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = steps, learning_rate = 2e-4, logging_steps = 5,
        optim = 'adamw_8bit', weight_decay = 0.01, lr_scheduler_type = 'linear', seed = 3407,
        output_dir = 'outputs', report_to = 'none',
    ),
)
trainer.train()

## 6 · Quick test

In [ ]:
FastLanguageModel.for_inference(model)
q = [{'role': 'system', 'content': "You are Mnemo, a helpful assistant that answers from the user's personal knowledge base."},
     {'role': 'user', 'content': 'Tell me something I taught you.'}]
ids = tokenizer.apply_chat_template(q, add_generation_prompt = True, return_tensors = 'pt').to('cuda')
out = model.generate(input_ids = ids, max_new_tokens = 200, temperature = 0.2)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens = True))

## 7 · Export as one GGUF file and download it (about 4.7 GB, takes a while)

In [ ]:
import glob, os, shutil
model.save_pretrained_gguf('mnemo-7b', tokenizer, quantization_method = 'q4_k_m')
g = sorted(glob.glob('**/*.gguf', recursive = True), key = os.path.getsize)[-1]
shutil.move(g, 'mnemo-7b-q4_k_m.gguf')
print('Ready:', os.path.getsize('mnemo-7b-q4_k_m.gguf') // 2**20, 'MB')
files.download('mnemo-7b-q4_k_m.gguf')

## Done
1. Move `mnemo-7b-q4_k_m.gguf` to **Downloads**.
2. Run `C:\xampp\htdocs\atlas\ai\Install trained Mnemo.bat`.
3. ATLAS → **Settings → Use it → Save**, then **Quality → Run benchmark** to compare with the old score.

If the browser download is slow or fails, copy it to Google Drive instead:
```python
from google.colab import drive; drive.mount('/content/drive')
!cp mnemo-7b-q4_k_m.gguf /content/drive/MyDrive/
```